# 📈 Retail Sales Forecasting & Demand Intelligence System

An end-to-end retail analytics project focused on sales forecasting, anomaly detection, and product demand segmentation using the Kaggle Superstore Sales dataset.

This project is organized into two parts:

- **Notebook** – Data preprocessing, exploratory data analysis, feature engineering, forecasting model development, anomaly detection, product segmentation, and model evaluation.
- **Streamlit Application** – An interactive dashboard that loads the generated models, predictions, and processed datasets for visualization and business insights.

> **Note:** Run the notebook from start to finish before launching the Streamlit application. The notebook generates all required models, processed datasets, forecasts, and reports used by the dashboard.


## 1. Project Introduction

### 1.1 Business Problem

Accurate sales forecasting is essential for retail businesses. Overstocking increases inventory costs, while understocking can lead to lost sales and poor customer experience.

The goal of this project is to analyze historical sales data and build a system that can:

- Forecast future sales at the overall, category, and region levels.
- Detect unusual sales patterns using anomaly detection techniques.
- Segment products based on demand characteristics to support inventory planning.
- Present the results through an interactive Streamlit dashboard.

---

### 1.2 Objectives

This project focuses on the following tasks:

- Perform data cleaning and feature engineering.
- Explore historical sales trends and seasonality.
- Build and compare multiple forecasting models.
- Forecast sales for different product categories and regions.
- Detect anomalous sales periods using statistical and machine learning methods.
- Segment products using clustering techniques.
- Save the generated models and processed data for deployment in a Streamlit application.

---

### 1.3 Dataset Overview

This project uses two datasets.

#### Primary Dataset

**Superstore Sales (`train.csv`)**

The primary dataset contains approximately four years of retail transaction data, including order dates, shipping dates, product categories, customer segments, regions, and sales information.

#### Secondary Dataset

**Video Game Sales (`vgsales.csv`)**

The secondary dataset is used to demonstrate anomaly detection on an independent time series and to practice working with multiple datasets in a single project.

### Dataset Location

Download the datasets from Kaggle and place them inside:

```text
data/raw/
```

Required files:

- `train.csv`
- `vgsales.csv`

---

### 1.4 Libraries Used

| Category | Libraries |
|----------|-----------|
| Data Analysis | `pandas`, `numpy` |
| Visualization | `matplotlib`, `seaborn`, `plotly` |
| Time Series | `statsmodels`, `prophet` |
| Machine Learning | `xgboost`, `scikit-learn` |
| Model Persistence | `joblib`, `json`, `pathlib` |

In [ ]:
# Core
import os
import json
import warnings
import joblib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Stats / time series
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ML models
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    PROPHET_AVAILABLE = False
    warnings.warn("Prophet is not installed. Run `pip install prophet` to enable the Prophet model.")

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 5)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Environment ready.")


In [ ]:
# ---------------------------------------------------------------------------
# Project paths (single source of truth, used throughout the notebook)
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd()
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"

# Artifacts are written directly into the sibling Streamlit app folder so the
# dashboard can consume them without any manual copy step. Adjust APP_DIR if
# your folder layout differs.
APP_DIR = PROJECT_ROOT.parent / "streamlit_app"
if not APP_DIR.exists():
    APP_DIR = PROJECT_ROOT  # fallback: keep artifacts alongside the notebook

MODELS_DIR = APP_DIR / "models"
PROCESSED_DIR = APP_DIR / "processed"
PREDICTIONS_DIR = APP_DIR / "predictions"
REPORTS_DIR = APP_DIR / "reports"

for d in [DATA_RAW_DIR, MODELS_DIR, PROCESSED_DIR, PREDICTIONS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SUPERSTORE_PATH = DATA_RAW_DIR / "train.csv"
VGSALES_PATH = DATA_RAW_DIR / "vgsales.csv"

print(f"Project root: {PROJECT_ROOT}")


## 2. Data Loading

The Superstore Sales dataset is loaded using Pandas, with the **Order Date** and **Ship Date** columns parsed as datetime values.

An initial inspection is performed to understand the dataset structure, including its shape, data types, missing values, duplicate records, and summary statistics.

> **Note:** Place `train.csv` inside the `data/raw/` directory before running this notebook.

In [ ]:
def generate_synthetic_superstore(n_rows: int = 9800, seed: int = RANDOM_STATE) -> pd.DataFrame:
    
    rng = np.random.default_rng(seed)
    start_date = pd.Timestamp("2015-01-01")
    end_date = pd.Timestamp("2018-12-31")
    order_dates = pd.to_datetime(
        rng.integers(start_date.value // 10**9, end_date.value // 10**9, n_rows), unit="s"
    )
    ship_lag = rng.integers(1, 8, n_rows)

    categories = {
        "Furniture": ["Bookcases", "Chairs", "Furnishings", "Tables"],
        "Office Supplies": ["Binders", "Paper", "Storage", "Art", "Labels", "Envelopes"],
        "Technology": ["Phones", "Accessories", "Machines", "Copiers"],
    }
    cat_choices = rng.choice(list(categories.keys()), n_rows, p=[0.22, 0.55, 0.23])
    sub_cat_choices = [rng.choice(categories[c]) for c in cat_choices]

    regions = ["East", "West", "Central", "South"]
    states_by_region = {
        "East": ["New York", "Pennsylvania", "Ohio"],
        "West": ["California", "Washington", "Oregon"],
        "Central": ["Texas", "Illinois", "Michigan"],
        "South": ["Florida", "Georgia", "North Carolina"],
    }
    region_choices = rng.choice(regions, n_rows)
    state_choices = [rng.choice(states_by_region[r]) for r in region_choices]

    segments = rng.choice(["Consumer", "Corporate", "Home Office"], n_rows, p=[0.5, 0.3, 0.2])

    # Base sales with yearly growth + monthly seasonality + noise
    month = order_dates.month
    year_growth = (order_dates.year - 2015) * 0.08
    seasonality = np.sin((month / 12) * 2 * np.pi) * 40 + (month.isin([11, 12])) * 90
    base = rng.gamma(shape=2.0, scale=90, size=n_rows)
    sales = base * (1 + year_growth) + seasonality + rng.normal(0, 15, n_rows)
    sales = np.clip(sales, 2, None).round(2)

    df = pd.DataFrame({
        "Row ID": np.arange(1, n_rows + 1),
        "Order ID": [f"US-{y}-{i:06d}" for y, i in zip(order_dates.year, range(n_rows))],
        "Order Date": order_dates,
        "Ship Date": order_dates + pd.to_timedelta(ship_lag, unit="D"),
        "Ship Mode": rng.choice(["Standard Class", "Second Class", "First Class", "Same Day"], n_rows),
        "Customer ID": [f"CUST-{i % 800:04d}" for i in range(n_rows)],
        "Segment": segments,
        "Country": "United States",
        "City": state_choices,
        "State": state_choices,
        "Region": region_choices,
        "Product ID": [f"PROD-{i % 1500:05d}" for i in range(n_rows)],
        "Category": cat_choices,
        "Sub-Category": sub_cat_choices,
        "Sales": sales,
    })
    return df.sort_values("Order Date").reset_index(drop=True)


def load_superstore_data(path: Path) -> pd.DataFrame:
    """Load the Superstore sales dataset, parsing date columns, with a synthetic fallback."""
    if path.exists():
        df = pd.read_csv(path, encoding="latin-1")
        source = "real Kaggle file"
    else:
        df = generate_synthetic_superstore()
        source = "synthetic fallback (place the real train.csv in data/raw/ for real results)"

    for col in ["Order Date", "Ship Date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)

    print(f"Loaded {len(df):,} rows from: {source}")
    return df


df = load_superstore_data(SUPERSTORE_PATH)
df.head()


In [ ]:
# ---- Structural inspection ----
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)


In [ ]:
df.describe(include=[np.number]).T

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print("Columns with missing values:")
print(missing if not missing.empty else "None")

duplicate_count = df.duplicated().sum()
print(f"\nFully duplicated rows: {duplicate_count}")

if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicates dropped. New shape:", df.shape)


### Loading the Secondary Dataset (Video Game Sales)

The **Video Game Sales** dataset is loaded for the anomaly detection section later in this notebook. It is used as an additional dataset to validate the anomaly detection techniques on a different sales domain.

In [ ]:
def load_vgsales_data(path: Path) -> pd.DataFrame:
    """Load the Video Game Sales dataset with a small synthetic fallback."""
    if path.exists():
        vg = pd.read_csv(path)
    else:
        rng = np.random.default_rng(RANDOM_STATE)
        years = rng.integers(1996, 2017, 1500)
        genres = rng.choice(
            ["Action", "Sports", "Shooter", "Role-Playing", "Racing", "Puzzle"], 1500
        )
        global_sales = np.clip(rng.gamma(1.5, 0.8, 1500) + (years > 2010) * 0.3, 0.01, None)
        vg = pd.DataFrame({"Year": years, "Genre": genres, "Global_Sales": global_sales})
    return vg


vg_df = load_vgsales_data(VGSALES_PATH)
vg_yearly = vg_df.dropna(subset=["Year"]).groupby("Year")["Global_Sales"].sum().reset_index()
vg_yearly["Year"] = vg_yearly["Year"].astype(int)
vg_yearly = vg_yearly.sort_values("Year")
vg_yearly.tail()


## 3. Feature Engineering

This section creates additional features from the existing dataset to improve exploratory analysis, forecasting, anomaly detection, and product segmentation.

Only information available in the original data is used during feature engineering.

In [ ]:
def add_time_features(data: pd.DataFrame, date_col: str = "Order Date") -> pd.DataFrame:
    """Add calendar-derived features from an order-date column."""
    out = data.copy()
    out["Year"] = out[date_col].dt.year
    out["Month"] = out[date_col].dt.month
    out["Week_Number"] = out[date_col].dt.isocalendar().week.astype(int)
    out["Quarter"] = out[date_col].dt.quarter
    out["Day_Of_Week"] = out[date_col].dt.day_name()
    out["Is_Weekend"] = out[date_col].dt.dayofweek.isin([5, 6])
    out["Is_Month_Start"] = out[date_col].dt.is_month_start
    out["Is_Month_End"] = out[date_col].dt.is_month_end

    season_map = {12: "Winter", 1: "Winter", 2: "Winter",
                  3: "Spring", 4: "Spring", 5: "Spring",
                  6: "Summer", 7: "Summer", 8: "Summer",
                  9: "Fall", 10: "Fall", 11: "Fall"}
    out["Season"] = out["Month"].map(season_map)
    return out


def add_shipping_features(data: pd.DataFrame) -> pd.DataFrame:
    """Add shipping-duration features from Order Date / Ship Date."""
    out = data.copy()
    if "Ship Date" in out.columns:
        out["Shipping_Days"] = (out["Ship Date"] - out["Order Date"]).dt.days
    return out


df = add_time_features(df)
df = add_shipping_features(df)
df[["Order Date", "Year", "Month", "Week_Number", "Quarter", "Season",
    "Day_Of_Week", "Is_Weekend", "Shipping_Days"]].head()


## 4. Exploratory Data Analysis

Before building forecasting models, it is important to understand the historical sales data.

This section analyzes sales across categories, regions, customer segments, and different time periods to identify trends, seasonality, and other business insights.

In [ ]:
revenue_by_category = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)

fig, ax = plt.subplots()
sns.barplot(x=revenue_by_category.values, y=revenue_by_category.index, ax=ax, palette="crest")
ax.set_title("Total Revenue by Category")
ax.set_xlabel("Revenue ($)")
plt.tight_layout()
plt.show()

revenue_by_category


In [ ]:
revenue_by_subcategory = df.groupby("Sub-Category")["Sales"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=revenue_by_subcategory.values, y=revenue_by_subcategory.index, ax=ax, palette="mako")
ax.set_title("Total Revenue by Sub-Category")
ax.set_xlabel("Revenue ($)")
plt.tight_layout()
plt.show()


In [ ]:
revenue_by_region = df.groupby("Region")["Sales"].sum().sort_values(ascending=False)
revenue_by_state = df.groupby("State")["Sales"].sum().sort_values(ascending=False).head(10)
revenue_by_segment = df.groupby("Segment")["Sales"].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(x=revenue_by_region.index, y=revenue_by_region.values, ax=axes[0], palette="viridis")
axes[0].set_title("Revenue by Region")

sns.barplot(x=revenue_by_state.values, y=revenue_by_state.index, ax=axes[1], palette="magma")
axes[1].set_title("Top 10 States by Revenue")

sns.barplot(x=revenue_by_segment.index, y=revenue_by_segment.values, ax=axes[2], palette="rocket")
axes[2].set_title("Revenue by Segment")

plt.tight_layout()
plt.show()


In [ ]:
monthly_sales = df.set_index("Order Date").resample("MS")["Sales"].sum()
weekly_sales = df.set_index("Order Date").resample("W")["Sales"].sum()
yearly_sales = df.set_index("Order Date").resample("YS")["Sales"].sum()

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
monthly_sales.plot(ax=axes[0], title="Monthly Sales", marker="o")
weekly_sales.plot(ax=axes[1], title="Weekly Sales", color="darkorange")
yearly_sales.plot(ax=axes[2], kind="bar", title="Yearly Sales", color="seagreen")
for ax in axes:
    ax.set_ylabel("Sales ($)")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots()
sns.boxplot(data=df, x="Shipping_Days", y="Ship Mode", ax=ax, orient="h", palette="Set2")
ax.set_title("Shipping Days Distribution by Ship Mode")
plt.tight_layout()
plt.show()

print("Average shipping days by region:")
print(df.groupby("Region")["Shipping_Days"].mean().round(2).sort_values())


In [ ]:
seasonal_avg = df.groupby("Season")["Sales"].mean().reindex(["Winter", "Spring", "Summer", "Fall"])
monthly_avg_by_year = df.groupby(["Year", "Month"])["Sales"].sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(x=seasonal_avg.index, y=seasonal_avg.values, ax=axes[0], palette="coolwarm")
axes[0].set_title("Average Order Value by Season")

sns.lineplot(data=monthly_avg_by_year, x="Month", y="Sales", hue="Year", marker="o", ax=axes[1], palette="tab10")
axes[1].set_title("Monthly Sales Pattern, Overlaid by Year (Seasonality Check)")
axes[1].set_xticks(range(1, 13))
plt.tight_layout()
plt.show()


In [ ]:
yoy_growth = yearly_sales.pct_change().dropna() * 100
print("Year-over-year revenue growth (%):")
print(yoy_growth.round(2))


### Key Observations

After completing the exploratory analysis, the following insights can be summarized:

- The category and sub-category analysis highlights the products contributing the highest revenue.
- Regional analysis helps identify areas with stronger sales performance and growth.
- Monthly sales trends reveal recurring seasonal patterns across different years.
- Shipping time analysis provides an overview of delivery performance across different regions and shipping modes.


## 5. Time Series Analysis

This section analyzes monthly sales trends to understand how sales change over time.

The analysis includes trend, seasonality, residual components, stationarity testing, and autocorrelation. These insights help in selecting appropriate forecasting models and tuning the SARIMA model.

In [ ]:
monthly_ts = df.set_index("Order Date").resample("MS")["Sales"].sum()
monthly_ts.index.freq = "MS"
monthly_ts.plot(title="Monthly Sales — Series Used for Forecasting", marker="o")
plt.ylabel("Sales ($)")
plt.tight_layout()
plt.show()


In [ ]:
decomposition = seasonal_decompose(monthly_ts, model="additive", period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True)
decomposition.observed.plot(ax=axes[0], title="Observed")
decomposition.trend.plot(ax=axes[1], title="Trend", color="darkorange")
decomposition.seasonal.plot(ax=axes[2], title="Seasonal", color="seagreen")
decomposition.resid.plot(ax=axes[3], title="Residual", color="firebrick", marker="o", linestyle="none")
plt.tight_layout()
plt.show()


### Observations

Time series decomposition helps understand the different patterns present in the sales data.

- **Trend** shows the long-term movement in sales.
- **Seasonality** captures recurring patterns across different time periods.
- **Residual** contains the unexplained variation after removing the trend and seasonal components. Large residual values may indicate unusual sales behavior.

In [ ]:
def run_adf_test(series: pd.Series, label: str = "series") -> dict:
    
    result = adfuller(series.dropna())
    output = {
        "adf_statistic": result[0],
        "p_value": result[1],
        "n_lags": result[2],
        "n_obs": result[3],
        "critical_values": result[4],
        "is_stationary": result[1] < 0.05,
    }
    verdict = "STATIONARY" if output["is_stationary"] else "NON-STATIONARY"
    print(f"ADF test on {label}: statistic={output['adf_statistic']:.3f}, "
          f"p-value={output['p_value']:.4f}  ->  {verdict}")
    print("  Plain English: a p-value below 0.05 means we can reject the hypothesis that the "
          "series has a unit root, i.e. its mean/variance are stable over time (stationary). "
          "A p-value above 0.05 means the series likely trends or drifts over time and should "
          "typically be differenced before fitting classical ARIMA-style models.")
    return output


adf_original = run_adf_test(monthly_ts, "original monthly sales")


In [ ]:
monthly_ts_diff = monthly_ts.diff().dropna()
adf_diff = run_adf_test(monthly_ts_diff, "first-differenced monthly sales")

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
monthly_ts.plot(ax=axes[0], title="Original Series")
monthly_ts_diff.plot(ax=axes[1], title="First Difference", color="darkorange")
plt.tight_layout()
plt.show()


In [ ]:
rolling_mean = monthly_ts.rolling(window=3).mean()
rolling_std = monthly_ts.rolling(window=3).std()

fig, ax = plt.subplots()
monthly_ts.plot(ax=ax, label="Sales", alpha=0.6)
rolling_mean.plot(ax=ax, label="Rolling Mean (3M)", color="darkorange")
rolling_std.plot(ax=ax, label="Rolling Std (3M)", color="firebrick")
ax.legend()
ax.set_title("Rolling Mean & Rolling Std (window = 3 months)")
plt.tight_layout()
plt.show()


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(monthly_ts.dropna(), ax=axes[0], lags=20)
plot_pacf(monthly_ts.dropna(), ax=axes[1], lags=20, method="ywm")
axes[0].set_title("ACF — Monthly Sales")
axes[1].set_title("PACF — Monthly Sales")
plt.tight_layout()
plt.show()


### ACF and PACF

The ACF and PACF plots help analyze the correlation between current sales and previous observations.

- **ACF** shows the overall correlation across different lag values.
- **PACF** highlights the direct influence of each lag.

The information from these plots is used to select suitable parameters for the SARIMA model and to identify seasonal patterns in the data.

## 6. Forecasting Models

This section compares three forecasting approaches using the monthly sales data:

- **SARIMA** – Statistical time series forecasting
- **Prophet** – Additive forecasting model
- **XGBoost** – Machine learning-based forecasting with lag features

Each model is evaluated using **MAE**, **RMSE**, and **MAPE**. The model with the lowest MAPE is selected for further forecasting and deployment.

### Train-Test Split

The last **3 months** of the dataset are reserved for testing, while the remaining data is used for training. This allows the models to be evaluated on unseen data before generating future forecasts.

In [ ]:
HORIZON = 3
train_ts = monthly_ts.iloc[:-HORIZON]
test_ts = monthly_ts.iloc[-HORIZON:]

print(f"Train range: {train_ts.index.min().date()} -> {train_ts.index.max().date()}  ({len(train_ts)} months)")
print(f"Test range:  {test_ts.index.min().date()} -> {test_ts.index.max().date()}  ({len(test_ts)} months)")


def evaluate_forecast(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Compute MAE, RMSE, MAPE, and R^2 for a forecast."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = float(np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1, y_true))) * 100)
    r2 = r2_score(y_true, y_pred) if len(y_true) > 1 else np.nan
    return {"MAE": round(mae, 2), "RMSE": round(rmse, 2), "MAPE": round(mape, 2), "R2": round(r2, 3)}


results = {}
forecasts = {}


### 6.1 Model 1 — SARIMA

SARIMA is used to model the trend and seasonal behavior present in the monthly sales data.

The model parameters were determined using the stationarity test, ACF/PACF plots, and parameter tuning. The best-performing configuration was selected based on the AIC score before generating the final forecast.

In [ ]:
import itertools

def grid_search_sarima(train: pd.Series, seasonal_period: int = 12) -> tuple:
    """Small AIC-based grid search around parameters suggested by the ACF/PACF/ADF diagnostics."""
    p = d = q = range(0, 2)
    P = D = Q = range(0, 2)
    best_aic = np.inf
    best_order = None
    best_seasonal_order = None

    for order in itertools.product(p, d, q):
        for seasonal_order in itertools.product(P, D, Q):
            try:
                model = SARIMAX(
                    train, order=order, seasonal_order=(*seasonal_order, seasonal_period),
                    enforce_stationarity=False, enforce_invertibility=False,
                )
                fit = model.fit(disp=False)
                if fit.aic < best_aic:
                    best_aic = fit.aic
                    best_order = order
                    best_seasonal_order = (*seasonal_order, seasonal_period)
            except Exception:
                continue
    return best_order, best_seasonal_order, best_aic


best_order, best_seasonal_order, best_aic = grid_search_sarima(train_ts)
print(f"Best SARIMA order: {best_order}, seasonal_order: {best_seasonal_order}, AIC: {best_aic:.2f}")

sarima_model = SARIMAX(
    train_ts, order=best_order, seasonal_order=best_seasonal_order,
    enforce_stationarity=False, enforce_invertibility=False,
).fit(disp=False)

sarima_forecast_result = sarima_model.get_forecast(steps=HORIZON)
sarima_forecast = sarima_forecast_result.predicted_mean
sarima_ci = sarima_forecast_result.conf_int()

results["SARIMA"] = evaluate_forecast(test_ts.values, sarima_forecast.values)
forecasts["SARIMA"] = sarima_forecast
print(results["SARIMA"])


In [ ]:
fig, ax = plt.subplots()
train_ts.plot(ax=ax, label="Train")
test_ts.plot(ax=ax, label="Actual (Test)", marker="o")
sarima_forecast.plot(ax=ax, label="SARIMA Forecast", marker="o")
ax.fill_between(sarima_ci.index, sarima_ci.iloc[:, 0], sarima_ci.iloc[:, 1], alpha=0.2, label="95% CI")
ax.legend()
ax.set_title("SARIMA — Actual vs Forecast")
plt.tight_layout()
plt.show()


### 6.2 Model 2 — Facebook Prophet

Prophet is a forecasting model developed for time series data with trend and seasonal patterns. It is trained using the same training dataset as the other models to ensure a consistent comparison.

Before training, the data is converted into Prophet's required format with the columns **`ds`** (date) and **`y`** (sales).

In [ ]:
prophet_train_df = train_ts.reset_index()
prophet_train_df.columns = ["ds", "y"]

if PROPHET_AVAILABLE:
    prophet_model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    prophet_model.fit(prophet_train_df)

    future = prophet_model.make_future_dataframe(periods=HORIZON, freq="MS")
    prophet_full_forecast = prophet_model.predict(future)
    prophet_forecast = prophet_full_forecast.set_index("ds")["yhat"].iloc[-HORIZON:]
    prophet_forecast.index.name = "Order Date"

    results["Prophet"] = evaluate_forecast(test_ts.values, prophet_forecast.values)
    forecasts["Prophet"] = prophet_forecast
    print(results["Prophet"])

    fig1 = prophet_model.plot(prophet_full_forecast)
    plt.title("Prophet Forecast — Trend + Seasonality")
    plt.show()

    fig2 = prophet_model.plot_components(prophet_full_forecast)
    plt.show()
else:
    print("Prophet not installed — skipping. `pip install prophet` to enable this model.")


### 6.3 Model 3 — XGBoost

Unlike SARIMA and Prophet, XGBoost treats forecasting as a supervised machine learning problem.

The model uses lag features, rolling statistics, and calendar-based features to learn historical sales patterns and predict future sales.

In [ ]:
def build_lag_features(ts: pd.Series) -> pd.DataFrame:
    """Build lag/rolling/calendar features for the XGBoost supervised approach."""
    feat = pd.DataFrame({"y": ts})
    for lag in [1, 2, 3, 6, 12]:
        feat[f"lag_{lag}"] = feat["y"].shift(lag)
    feat["rolling_mean_3"] = feat["y"].shift(1).rolling(3).mean()
    feat["rolling_std_3"] = feat["y"].shift(1).rolling(3).std()
    feat["month"] = feat.index.month
    feat["quarter"] = feat.index.quarter
    season_map = {12: 0, 1: 0, 2: 0, 3: 1, 4: 1, 5: 1, 6: 2, 7: 2, 8: 2, 9: 3, 10: 3, 11: 3}
    feat["season"] = feat["month"].map(season_map)
    return feat.dropna()


full_features = build_lag_features(monthly_ts)
feature_cols = [c for c in full_features.columns if c != "y"]

xgb_train = full_features.iloc[:-HORIZON]
xgb_test = full_features.iloc[-HORIZON:]

xgb_model = xgb.XGBRegressor(
    n_estimators=300, max_depth=3, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, random_state=RANDOM_STATE,
)
xgb_model.fit(xgb_train[feature_cols], xgb_train["y"])

xgb_pred = pd.Series(xgb_model.predict(xgb_test[feature_cols]), index=xgb_test.index)

results["XGBoost"] = evaluate_forecast(xgb_test["y"].values, xgb_pred.values)
forecasts["XGBoost"] = xgb_pred
print(results["XGBoost"])


In [ ]:
fig, ax = plt.subplots()
monthly_ts.plot(ax=ax, label="Actual", alpha=0.7)
xgb_pred.plot(ax=ax, label="XGBoost Forecast", marker="o")
ax.legend()
ax.set_title("XGBoost — Actual vs Forecast")
plt.tight_layout()
plt.show()

importances = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values()
importances.plot(kind="barh", title="XGBoost Feature Importance")
plt.tight_layout()
plt.show()


### 6.4 Model Comparison

The performance of all three forecasting models is evaluated using the same test dataset.

The comparison is based on **MAE**, **RMSE**, and **MAPE**. The model with the lowest MAPE is selected for category-level and region-level forecasting in the following sections.


In [ ]:
comparison_rows = []
for name, metrics in results.items():
    row = {"Model": name, **metrics}
    fc = forecasts[name]
    for i, val in enumerate(fc.values, start=1):
        row[f"Forecast_Month_{i}"] = round(float(val), 2)
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).set_index("Model")
comparison_df


In [ ]:
best_model_name = comparison_df["MAPE"].idxmin()
print(f"Best model selected automatically (lowest MAPE): {best_model_name}")
print(comparison_df.loc[best_model_name])


### Model Selection

The best forecasting model is selected based on its evaluation metrics.

Although **MAE**, **RMSE**, and **MAPE** are reported for every model, **MAPE** is used as the primary metric for model selection because it provides an easy-to-interpret percentage error.

In [ ]:
# Persist forecasts for the dashboard immediately
comparison_df.to_csv(PREDICTIONS_DIR / "model_comparison.csv")

overall_forecast_df = pd.DataFrame({name: fc for name, fc in forecasts.items()})
overall_forecast_df["Actual"] = test_ts
overall_forecast_df.to_csv(PREDICTIONS_DIR / "overall_forecast.csv")

with open(REPORTS_DIR / "best_model.json", "w") as f:
    json.dump({"best_model": best_model_name, "selected_on": "MAPE",
               "metrics": results[best_model_name]}, f, indent=2)

print("Saved model_comparison.csv, overall_forecast.csv, best_model.json")


## 7. Forecasting by Category and Region

After selecting the best forecasting model, the same approach is applied to different product categories and regions.

Separate forecasts are generated for **Furniture**, **Technology**, **Office Supplies**, **East**, and **West** to compare future sales trends across these business segments.

In [ ]:
def monthly_series_for(data: pd.DataFrame, column: str, value: str) -> pd.Series:
    """Aggregate monthly sales for one value of a Category/Region column."""
    subset = data[data[column] == value]
    ts = subset.set_index("Order Date").resample("MS")["Sales"].sum()
    ts.index.freq = "MS"
    return ts


def forecast_segment(ts: pd.Series, horizon: int = HORIZON) -> tuple:
    """Fit the lag-feature XGBoost approach on a segment series and forecast `horizon` months ahead."""
    feat = build_lag_features(ts)
    if len(feat) < horizon + 6:
        return None, None  # not enough history for a meaningful lag model

    train_feat = feat.iloc[:-horizon]
    test_feat = feat.iloc[-horizon:]

    model = xgb.XGBRegressor(
        n_estimators=200, max_depth=3, learning_rate=0.05, random_state=RANDOM_STATE
    )
    model.fit(train_feat[feature_cols], train_feat["y"])
    pred = pd.Series(model.predict(test_feat[feature_cols]), index=test_feat.index)
    metrics = evaluate_forecast(test_feat["y"].values, pred.values)
    return pred, metrics


segments_to_forecast = {
    "Category": ["Furniture", "Technology", "Office Supplies"],
    "Region": ["East", "West"],
}

segment_forecasts = {}
segment_metrics = {}

for column, values in segments_to_forecast.items():
    for value in values:
        ts_seg = monthly_series_for(df, column, value)
        pred, metrics = forecast_segment(ts_seg)
        segment_forecasts[value] = pred
        segment_metrics[value] = metrics
        if pred is not None:
            print(f"{value}: {metrics}")
        else:
            print(f"{value}: not enough history for a reliable segment forecast")


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for name, pred in segment_forecasts.items():
    if pred is not None:
        pred.plot(ax=ax, marker="o", label=name)
ax.set_title("Segment-Level Forecasts — Next 3 Months (Category & Region)")
ax.legend()
plt.tight_layout()
plt.show()

segment_metrics_df = pd.DataFrame(segment_metrics).T
segment_metrics_df.to_csv(PREDICTIONS_DIR / "segment_forecast_metrics.csv")

segment_forecast_df = pd.DataFrame({k: v for k, v in segment_forecasts.items() if v is not None})
segment_forecast_df.to_csv(PREDICTIONS_DIR / "segment_forecasts.csv")
segment_metrics_df


### Key Findings

The category and region forecasts are analyzed to identify segments with the highest expected sales growth over the next three months. These insights can help support inventory planning and business decision-making.

## 8. Anomaly Detection

Two different approaches are used to detect unusual sales patterns in the weekly sales data.

- **Isolation Forest** identifies anomalies using a machine learning approach.
- **Rolling Z-Score** identifies anomalies based on statistical deviation from recent sales trends.

Comparing the results from both methods provides a more reliable view of unusual sales events.

In [ ]:
weekly_ts = df.set_index("Order Date").resample("W")["Sales"].sum()

anomaly_features = pd.DataFrame({
    "sales": weekly_ts,
    "rolling_mean": weekly_ts.rolling(4, min_periods=1).mean(),
    "rolling_std": weekly_ts.rolling(4, min_periods=1).std().fillna(0),
})

iso_forest = IsolationForest(contamination=0.05, random_state=RANDOM_STATE)
anomaly_features["iso_flag"] = iso_forest.fit_predict(anomaly_features[["sales", "rolling_mean", "rolling_std"]])
anomaly_features["is_anomaly_iso"] = anomaly_features["iso_flag"] == -1

rolling_mean_4 = weekly_ts.rolling(4, min_periods=1).mean()
rolling_std_4 = weekly_ts.rolling(4, min_periods=1).std().fillna(0)
z_scores = (weekly_ts - rolling_mean_4) / rolling_std_4.replace(0, np.nan)
anomaly_features["z_score"] = z_scores
anomaly_features["is_anomaly_zscore"] = anomaly_features["z_score"].abs() > 2

anomaly_features["is_anomaly_both"] = anomaly_features["is_anomaly_iso"] & anomaly_features["is_anomaly_zscore"]

print(f"Isolation Forest flagged: {anomaly_features['is_anomaly_iso'].sum()} weeks")
print(f"Z-Score method flagged:   {anomaly_features['is_anomaly_zscore'].sum()} weeks")
print(f"Both methods agree on:    {anomaly_features['is_anomaly_both'].sum()} weeks")


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
weekly_ts.plot(ax=ax, label="Weekly Sales", alpha=0.7)

iso_points = anomaly_features[anomaly_features["is_anomaly_iso"]]
z_points = anomaly_features[anomaly_features["is_anomaly_zscore"]]

ax.scatter(iso_points.index, iso_points["sales"], color="red", label="Isolation Forest Anomaly", zorder=5)
ax.scatter(z_points.index, z_points["sales"], color="orange", marker="x", s=80, label="Z-Score Anomaly", zorder=5)
ax.legend()
ax.set_title("Weekly Sales — Detected Anomalies")
plt.tight_layout()
plt.show()


In [ ]:
anomaly_report = anomaly_features[anomaly_features["is_anomaly_iso"] | anomaly_features["is_anomaly_zscore"]].copy()
anomaly_report["month_name"] = anomaly_report.index.month_name()

def likely_reason(row) -> str:
    """A simple heuristic explanation generator based on calendar position and direction of deviation."""
    if row["month_name"] in ["November", "December"]:
        return "Likely holiday/Black Friday/year-end promotional spike"
    if row["sales"] > row["rolling_mean"]:
        return "Above-trend spike — possible bulk order, promotion, or data entry duplication"
    return "Below-trend dip — possible stockout, logistics disruption, or reporting gap"

anomaly_report["likely_business_reason"] = anomaly_report.apply(likely_reason, axis=1)
anomaly_report = anomaly_report.reset_index().rename(columns={"index": "Week"})
anomaly_report.to_csv(PREDICTIONS_DIR / "anomalies.csv", index=False)
anomaly_report[["Week", "sales", "z_score", "is_anomaly_iso", "is_anomaly_zscore", "likely_business_reason"]]


### Comparison of Detection Methods

The results from both anomaly detection methods are compared to understand their similarities and differences.

Weeks identified by both methods are treated as stronger anomaly candidates, while unique detections provide additional insights into unusual sales behavior.

---

### Secondary Dataset Analysis

The rolling Z-Score method is also applied to the **Video Game Sales** dataset to evaluate anomaly detection on a separate sales dataset.

In [ ]:
vg_ts = vg_yearly.set_index("Year")["Global_Sales"]
vg_rolling_mean = vg_ts.rolling(3, min_periods=1).mean()
vg_rolling_std = vg_ts.rolling(3, min_periods=1).std().fillna(0)
vg_z = (vg_ts - vg_rolling_mean) / vg_rolling_std.replace(0, np.nan)
vg_anomalies = vg_ts[vg_z.abs() > 2]

fig, ax = plt.subplots()
vg_ts.plot(ax=ax, marker="o", label="Global Video Game Sales")
ax.scatter(vg_anomalies.index, vg_anomalies.values, color="red", zorder=5, label="Anomaly (Z-Score)")
ax.legend()
ax.set_title("Video Game Sales — Anomaly Detection Practice")
plt.tight_layout()
plt.show()


## 9. Product Demand Segmentation

Product sub-categories are grouped into different demand segments using **K-Means clustering**.

The clusters are visualized with **PCA**, and each segment is analyzed to identify its demand pattern and recommend an appropriate inventory strategy.


In [ ]:
def build_subcategory_features(data: pd.DataFrame) -> pd.DataFrame:
    """Aggregate order-level data into one row per Sub-Category with demand-behaviour features."""
    monthly_by_sub = (
        data.set_index("Order Date")
        .groupby("Sub-Category")
        .resample("MS")["Sales"].sum()
        .reset_index()
    )

    agg = data.groupby("Sub-Category").agg(
        total_sales=("Sales", "sum"),
        order_count=("Sales", "count"),
        avg_order_value=("Sales", "mean"),
    )

    monthly_stats = monthly_by_sub.groupby("Sub-Category")["Sales"].agg(
        monthly_mean="mean", monthly_std="std"
    ).fillna(0)

    def growth_rate(group: pd.DataFrame) -> float:
        yearly = group.set_index("Order Date").resample("YS")["Sales"].sum()
        if len(yearly) < 2 or yearly.iloc[0] == 0:
            return 0.0
        return float((yearly.iloc[-1] - yearly.iloc[0]) / yearly.iloc[0] * 100)

    growth = data.groupby("Sub-Category").apply(growth_rate).rename("growth_rate_pct")

    features = agg.join(monthly_stats).join(growth)
    features["sales_volatility"] = features["monthly_std"]
    return features.reset_index()


subcat_features = build_subcategory_features(df)
subcat_features


In [ ]:
feature_cols_cluster = ["total_sales", "growth_rate_pct", "sales_volatility", "avg_order_value",
                        "monthly_mean", "monthly_std"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(subcat_features[feature_cols_cluster])

inertias = []
k_range = range(2, min(8, len(subcat_features)))
for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure()
plt.plot(list(k_range), inertias, marker="o")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal k")
plt.tight_layout()
plt.show()


In [ ]:
OPTIMAL_K = 4  # chosen from the elbow plot above; adjust after inspecting the real dataset's curve

kmeans_model = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=10)
subcat_features["cluster"] = kmeans_model.fit_predict(X_scaled)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_coords = pca.fit_transform(X_scaled)
subcat_features["pca_1"] = pca_coords[:, 0]
subcat_features["pca_2"] = pca_coords[:, 1]

fig = px.scatter(
    subcat_features, x="pca_1", y="pca_2", color=subcat_features["cluster"].astype(str),
    text="Sub-Category", title="Product Sub-Category Clusters (PCA-reduced)",
    labels={"color": "Cluster"},
)
fig.update_traces(textposition="top center")
fig.show()


In [ ]:
def label_cluster(row: pd.Series, medians: pd.Series) -> str:
    """Translate cluster statistics into a business-readable demand label."""
    high_volume = row["total_sales"] >= medians["total_sales"]
    high_volatility = row["sales_volatility"] >= medians["sales_volatility"]
    growing = row["growth_rate_pct"] >= medians["growth_rate_pct"]

    if high_volume and not high_volatility:
        return "High Volume, Stable Demand"
    if not high_volume and high_volatility:
        return "Low Volume, High Volatility"
    if growing:
        return "Growing Demand"
    return "Declining / Niche Demand"


cluster_profile = subcat_features.groupby("cluster")[feature_cols_cluster].median()
global_medians = subcat_features[feature_cols_cluster].median()
cluster_profile["business_label"] = cluster_profile.apply(lambda r: label_cluster(r, global_medians), axis=1)

recommendation_map = {
    "High Volume, Stable Demand": "Maintain steady safety stock; automate reordering; low forecast risk.",
    "Low Volume, High Volatility": "Keep lean stock with a fast-reorder supplier; avoid large bulk buys.",
    "Growing Demand": "Increase stock ahead of trend; monitor weekly; consider expanding SKU range.",
    "Declining / Niche Demand": "Reduce stock commitment; consider markdown or discontinuation review.",
}
cluster_profile["stocking_recommendation"] = cluster_profile["business_label"].map(recommendation_map)

subcat_features = subcat_features.merge(
    cluster_profile[["business_label", "stocking_recommendation"]], on="cluster", how="left"
)

subcat_features.to_csv(PROCESSED_DIR / "subcategory_segments.csv", index=False)
cluster_profile


## 10. Model Saving

After training and evaluation, the generated models, processed datasets, predictions, and reports are saved for deployment in the Streamlit dashboard.

This allows the dashboard to display results without retraining the models.

In [ ]:
# Best overall model object (re-trained on the FULL series so production forecasts use all history)
if best_model_name == "XGBoost":
    full_feat = build_lag_features(monthly_ts)
    final_model = xgb.XGBRegressor(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, random_state=RANDOM_STATE,
    )
    final_model.fit(full_feat[feature_cols], full_feat["y"])
    joblib.dump(final_model, MODELS_DIR / "best_model_xgboost.joblib")

elif best_model_name == "SARIMA":
    final_model = SARIMAX(
        monthly_ts, order=best_order, seasonal_order=best_seasonal_order,
        enforce_stationarity=False, enforce_invertibility=False,
    ).fit(disp=False)
    final_model.save(str(MODELS_DIR / "best_model_sarima.pkl"))

elif best_model_name == "Prophet" and PROPHET_AVAILABLE:
    full_prophet_df = monthly_ts.reset_index()
    full_prophet_df.columns = ["ds", "y"]
    final_model = Prophet(yearly_seasonality=True)
    final_model.fit(full_prophet_df)
    joblib.dump(final_model, MODELS_DIR / "best_model_prophet.joblib")

joblib.dump(scaler, MODELS_DIR / "feature_scaler.joblib")
joblib.dump(kmeans_model, MODELS_DIR / "kmeans_cluster_model.joblib")
joblib.dump(pca, MODELS_DIR / "pca_model.joblib")
joblib.dump(iso_forest, MODELS_DIR / "isolation_forest_model.joblib")

print("Models saved to:", MODELS_DIR)


In [ ]:
# Processed datasets used by the dashboard
df.to_csv(PROCESSED_DIR / "cleaned_transactions.csv", index=False)
monthly_ts.to_csv(PROCESSED_DIR / "monthly_sales.csv", header=["Sales"])
weekly_ts.to_csv(PROCESSED_DIR / "weekly_sales.csv", header=["Sales"])
anomaly_features.reset_index().rename(columns={"index": "Week"}).to_csv(
    PROCESSED_DIR / "weekly_sales_with_anomaly_flags.csv", index=False
)

print("Processed datasets saved to:", PROCESSED_DIR)
print("Prediction files saved to:", PREDICTIONS_DIR)

# Final manifest so the app can sanity-check what is available at startup
manifest = {
    "generated_at": datetime.now().isoformat(),
    "best_model": best_model_name,
    "model_files": [p.name for p in MODELS_DIR.glob("*")],
    "processed_files": [p.name for p in PROCESSED_DIR.glob("*")],
    "prediction_files": [p.name for p in PREDICTIONS_DIR.glob("*")],
}
with open(REPORTS_DIR / "artifact_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

manifest


## 11. Business Insights

### Key Findings

- The forecasting models were evaluated using **MAE**, **RMSE**, and **MAPE** to identify the most accurate approach for this dataset.
- The sales data exhibits clear seasonal patterns, indicating that demand changes consistently during specific periods of the year.
- Forecasts generated for different product categories and regions highlight variations in expected demand, which can support inventory and resource planning.
- Anomaly detection identified several weeks with unusual sales activity that may be associated with promotions, inventory issues, or unexpected business events.
- Product segmentation grouped sub-categories with similar demand characteristics, making it easier to define inventory strategies for different types of products.

---

### Business Recommendations

1. Use the best-performing forecasting model for short-term inventory planning and replenishment decisions.
2. Monitor categories and regions showing consistent growth to ensure sufficient stock availability during periods of increased demand.
3. Review detected anomalies regularly to distinguish between genuine business events and potential data quality issues.
4. Apply different inventory strategies for each demand segment instead of using a single stocking policy across all products.

---

### Project Limitations

- The forecasting models rely only on historical sales data and do not consider external factors such as promotions, pricing changes, holidays, or economic conditions.
- Forecast accuracy may decrease if future sales patterns differ significantly from historical trends.
- The anomaly detection methods identify unusual observations but do not explain the exact business reason behind each anomaly.

---

### Future Improvements

- Incorporate external variables such as promotions, holidays, and pricing information into the forecasting models.
- Evaluate deep learning approaches such as **LSTM** or **Temporal Fusion Transformer (TFT)** for long-term forecasting.
- Build an automated model retraining pipeline to keep forecasts up to date as new sales data becomes available.
- Deploy the forecasting pipeline using **FastAPI**, **Docker**, and a cloud platform for production use.